# SR-DAS, revised: pre-registered test on a clean holdout (notebook 3)

This notebook is **self-contained**: it downloads its own data and contains all code. It does not need notebooks 1 or 2.

### Why a revision
The original specification (θ = 0.5) was tested on the 49 industries (1990–2025) and on S&P 500 stocks (2005–2025). In both, it behaved almost identically to simply picking the top-quality assets (daily return correlation 0.96–0.98). The reason is a **scale mismatch**: in the objective `2θ Σ z_i + log det K_S`, the quality term varied 3–5 times more across candidate sets than the diversity term, so diversification barely influenced the choice. Pure diversity (θ = 0) did the opposite: the lowest crash-time correlation but the highest volatility, because the least-correlated assets are often the most volatile ones.

### What is new
1. **B-SR-DAS (primary)** — θ is set at each rebalance by a fixed rule that gives the quality and diversity terms *equal spread*. Nothing is tuned on returns.
2. **S-SR-DAS (secondary)** — screen the top 50% by quality, then choose by pure diversity.
3. **Trade-off frontier** — fixed θ values from 0 to 1, with all baselines as points.

### Design that protects against data snooping
* **Holdout: 1970–1989** (49 industries). This period was **not used in any earlier analysis**. Its results are the headline.
* **Replication: 1990–latest.** Seen during development, so it is supporting evidence only.
* The specification and seven hypotheses are **frozen in the next section** and fingerprinted before any result is computed.

**How to run:** `Runtime → Run all`. About 10–15 minutes on a standard Colab CPU (the frontier section is the slowest; it can be switched off).

## 0. Setup

In [ ]:
!pip -q install --upgrade networkx scikit-learn > /dev/null

In [ ]:
import os
from IPython.display import display
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100); pd.set_option("display.max_colwidth", 90)

# ---- run options (do not affect the pre-registered specification) ----
RUN_REPLICATION = True        # 1990-latest replication
RUN_FRONTIER = True           # fixed-theta trade-off frontier (slowest part)
N_BOOT = 2000                 # bootstrap replications for tests and CIs
N_RANDOM = 20                 # random-k baskets
SAVE_TO_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/srdas/revised"
OUT_DIR = "outputs_revised"
DATA_DIR = "data"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = os.path.join(DRIVE_DIR, "data")

## 1. Pre-registered specification
Run once and keep the printed fingerprint with your notes. The specification is also saved to `outputs_revised/specification.json`.

In [ ]:
import json, hashlib, datetime

# =====================================================================
#  PRE-REGISTERED SPECIFICATION — frozen before the holdout is run.
#  Do NOT edit after the first run; any change must be reported as a deviation.
# =====================================================================
SPEC = {
    "primary_method": "B-SR-DAS (balanced): log-det on robust stress correlation, theta set by balanced_theta (beta=1)",
    "secondary_method": "S-SR-DAS: top 50% by quality, then pure log-det (theta=0) on robust stress correlation",
    "dataset": "Fama-French 49 industry portfolios, daily, value-weighted",
    "holdout_period": ["1970-01-01", "1989-12-31"],
    "replication_period": ["1990-01-01", "latest"],
    "k": 10, "lookback_days": 1260, "rebalance_days": 63, "stress_q": 0.10,
    "sub_len": 504, "sub_step": 126, "agg_q": 0.75, "kernel_a": 0.5, "cost_bps": 10,
    "weights": "equal",
    "hypotheses": {
        "H1": "Primary's out-of-sample stress-day basket correlation < Top-k Quality (paired Wilcoxon, one-sided, alpha=0.05)",
        "H2": "Primary's volatility < EW-All 1/N (block-bootstrap 95% CI of ratio entirely below 1)",
        "H3": "Primary's volatility < pure-diversity SR-DAS theta=0 (95% CI of ratio below 1)",
        "H4": "Primary's Sharpe not significantly below EW-All 1/N (two-sided block bootstrap, alpha=0.05)",
        "H5": "RQ1: stress-day basket correlation < Stress-Cluster rule (one-sided Wilcoxon)",
        "H6": "RQ2: stress-day basket correlation < same method with all-day correlation (one-sided Wilcoxon)",
        "H7": "RQ3: stress-day basket correlation < same method with single-window estimate (one-sided Wilcoxon)",
    },
    "multiple_testing": "Holm correction across all Sharpe comparisons of the primary method",
    "decision_rule": "The holdout verdicts are the headline results; the 1990+ replication was seen during development and is reported as supporting evidence only.",
}
SPEC_JSON = json.dumps(SPEC, indent=2, sort_keys=True)
SPEC_HASH = hashlib.sha256(SPEC_JSON.encode()).hexdigest()[:16]
print(SPEC_JSON)
print("\nSpecification fingerprint:", SPEC_HASH)


## 2. Library
All code needed by this notebook. Nothing here downloads data.

### 2.1 Imports

In [ ]:
import math
import time
import warnings
from dataclasses import dataclass, field
from functools import cached_property

import numpy as np
import pandas as pd
import networkx as nx
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=RuntimeWarning)
TRADING_DAYS = 252


### 2.2 Dependence estimators
Ledoit–Wolf shrunk correlation; the stress-conditioned, temporally robust correlation (stress days only, overlapping sub-windows, pessimistic `agg_q` quantile per pair, PSD projection); empirical lower tail dependence (baseline).

In [ ]:
def standardize(X):
    """Column-standardize a T x N array (zero-variance columns left at 0)."""
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=1)
    sd = np.where(sd > 1e-12, sd, 1.0)
    return (X - mu) / sd


def shrunk_corr(X):
    """Ledoit-Wolf shrunk correlation matrix of a T x N return array.

    Returns are standardized first, so Ledoit-Wolf shrinks the sample
    correlation toward the identity. Works when T < N (as on stress days).
    """
    Z = standardize(np.asarray(X, dtype=float))
    S = LedoitWolf().fit(Z).covariance_
    d = np.sqrt(np.clip(np.diag(S), 1e-12, None))
    C = S / np.outer(d, d)
    np.fill_diagonal(C, 1.0)
    return C


def nearest_psd_corr(C, eps=1e-6):
    """Project a symmetric matrix to a valid (PSD, unit-diagonal) correlation matrix.

    Eigenvalue clipping followed by rescaling to unit diagonal. Rescaling is a
    congruence transform, so positive semi-definiteness is preserved.
    """
    C = (C + C.T) / 2.0
    w, V = np.linalg.eigh(C)
    w = np.clip(w, eps, None)
    C2 = (V * w) @ V.T
    d = np.sqrt(np.diag(C2))
    C2 = C2 / np.outer(d, d)
    C2 = (C2 + C2.T) / 2.0
    np.fill_diagonal(C2, 1.0)
    return C2


def stress_mask(mkt, stress_q):
    """Boolean mask of 'stress days': market return at or below its stress_q quantile."""
    thr = np.quantile(mkt, stress_q)
    return mkt <= thr


def window_starts(T, sub_len, sub_step):
    """Start indices of sub-windows of length sub_len, anchored at the most recent day."""
    if sub_len >= T:
        return [0]
    starts = []
    s = T - sub_len
    while s >= 0:
        starts.append(s)
        s -= sub_step
    return starts[::-1]


def robust_stress_corr(R, mkt, stress_q=0.10, sub_len=504, sub_step=126,
                       agg_q=0.75, min_stress_days=20):
    """Stress-conditioned, temporally robust correlation matrix (the SR-DAS dependence estimate).

    1. Split the look-back into overlapping sub-windows.
    2. In each sub-window keep only stress days (market in its bottom stress_q tail)
       and estimate a Ledoit-Wolf shrunk correlation on those days.
    3. Aggregate pessimistically: for every pair take the agg_q quantile of its
       stress correlation across sub-windows (agg_q > 0.5 = assume the pair is
       as dependent as it has reliably been in bad times).
    4. Project to the nearest valid correlation matrix so log-det is well defined.

    R: T x N array of returns, mkt: length-T array of market returns.
    Returns (C, n_windows_used).
    """
    R = np.asarray(R, dtype=float)
    mkt = np.asarray(mkt, dtype=float)
    T = R.shape[0]
    mats = []
    for s in window_starts(T, sub_len, sub_step):
        e = min(T, s + sub_len)
        m = stress_mask(mkt[s:e], stress_q)
        if m.sum() < min_stress_days:
            continue
        mats.append(shrunk_corr(R[s:e][m]))
    if not mats:  # fall back to all-day correlation
        return shrunk_corr(R), 0
    stack = np.stack(mats)
    C = np.quantile(stack, agg_q, axis=0) if len(mats) > 1 else stack[0]
    return nearest_psd_corr(C), len(mats)


def lower_tail_dependence(R, u=0.10):
    """Empirical (non-parametric) lower tail-dependence coefficients.

    lambda_ij = P(U_i <= u, U_j <= u) / u, with U the column ranks scaled to (0,1).
    This is the copula-free estimator used in tail-dependence clustering work.
    """
    R = np.asarray(R, dtype=float)
    T = R.shape[0]
    U = (pd.DataFrame(R).rank(axis=0).values) / (T + 1.0)
    I = (U <= u).astype(float)
    lam = (I.T @ I) / T / u
    lam = np.clip(lam, 0.0, 1.0)
    np.fill_diagonal(lam, 1.0)
    return lam


### 2.3 Quality score
Low volatility and 12-1 momentum, rank-normalized and averaged into a z-score. Look-back data only.

In [ ]:
def rank_normal(x):
    """Map a cross-section to standard-normal scores via ranks (robust to outliers)."""
    x = pd.Series(x)
    r = x.rank(method="average")
    return pd.Series(norm.ppf((r - 0.5) / len(x)), index=x.index)


def quality_scores(R_df, vol_window=252, mom_window=252, mom_skip=21):
    """Composite quality score z_i from price-only signals with out-of-sample evidence.

    - Low volatility: minus the standard deviation of the last vol_window daily returns.
    - 12-1 momentum: return from t-mom_window to t-mom_skip (skips the last month).
    Each signal is rank-normalized, averaged and re-normalized to a z-score.
    Uses only data up to the rebalance date (no look-ahead).
    """
    X = R_df.values
    vol = X[-vol_window:].std(axis=0, ddof=1)
    seg = X[-mom_window:-mom_skip] if mom_skip > 0 else X[-mom_window:]
    mom = np.prod(1.0 + seg, axis=0) - 1.0
    z = (rank_normal(-vol).values + rank_normal(mom).values) / 2.0
    z = (z - z.mean()) / (z.std(ddof=1) + 1e-12)
    return pd.Series(z, index=R_df.columns)


### 2.4 Selection rules (original method and baselines)
Greedy log-det selection with kernel `K = (1-a)C + a·11ᵀ` (negative correlation counts as diverse), cluster-then-pick, MST periphery (Pozzi et al. 2013), PCA-KMeans (Vo et al. 2026).

In [ ]:
def affine_corr_kernel(C, a=0.5):
    """Similarity kernel K = (1-a) C + a 11^T.

    Maps correlation -1..1 to similarity 0..1 for a=0.5, so that NEGATIVE
    correlation counts as MORE diverse (a plain correlation kernel would
    penalize hedging pairs, since det([[1,r],[r,1]]) = 1-r^2).
    K is PSD because it is a sum of PSD matrices. One can show
    det(K_S) = (1-a)^k det(C_S) (1 + a/(1-a) * 1' C_S^{-1} 1),
    i.e. it rewards both low generalized correlation and a low-risk
    minimum-variance combination of the chosen assets.
    """
    return (1.0 - a) * C + a * np.ones_like(C)


def greedy_logdet(L, k):
    """Greedy MAP for log det(L_S) with |S| = k (fast incremental Cholesky).

    log det is submodular, so greedy has the classic (1 - 1/e) guarantee for
    the monotone case and is the standard DPP MAP heuristic otherwise.
    Returns the list of selected indices (in order of selection).
    """
    N = L.shape[0]
    k = min(k, N)
    cis = np.zeros((k, N))
    di2 = np.diag(L).astype(float).copy()
    sel = []
    for j in range(k):
        score = di2.copy()
        score[sel] = -np.inf
        i = int(np.argmax(score))
        if di2[i] <= 1e-12:
            break
        sel.append(i)
        ei = (L[i, :] - cis[:j, i] @ cis[:j, :]) / np.sqrt(di2[i])
        cis[j, :] = ei
        di2 = di2 - ei ** 2
    return sel


def logdet_select(C, z, k, theta=0.5, a=0.5):
    """SR-DAS selection rule on a given correlation estimate C and quality z.

    L = diag(q) K diag(q) with q = exp(theta * z), so that
    log det L_S = 2 theta sum_{i in S} z_i + log det K_S
    (theta trades quality against diversity; theta=0 is pure diversity).
    """
    K = affine_corr_kernel(C, a)
    q = np.exp(theta * np.asarray(z))
    L = K * np.outer(q, q)
    return greedy_logdet(L, k)


def cluster_pick(D, z, k, method="average"):
    """Cluster-then-pick baseline: hierarchical clustering on distance D into k
    clusters, then the highest-quality asset from each cluster."""
    D = np.asarray(D, dtype=float)
    D = (D + D.T) / 2.0
    np.fill_diagonal(D, 0.0)
    D = np.clip(D, 0.0, None)
    Z = linkage(squareform(D, checks=False), method=method)
    labels = fcluster(Z, t=k, criterion="maxclust")
    z = np.asarray(z)
    sel = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        sel.append(int(idx[np.argmax(z[idx])]))
    # fcluster can return fewer than k clusters on ties; top up by quality
    if len(sel) < k:
        rest = [i for i in np.argsort(-z) if i not in sel]
        sel += rest[: k - len(sel)]
    return sel[:k]


def corr_distance(C):
    return np.sqrt(np.clip(2.0 * (1.0 - C), 0.0, None))


def mst_periphery_select(C, k):
    """Pozzi, Di Matteo & Aste (2013)-style selection: build the Minimum Spanning
    Tree on correlation distance and pick the k most peripheral nodes.

    Hybrid centrality = rank(degree) + rank(betweenness) + rank(closeness)
    + rank(-eccentricity); lowest values are the most peripheral.
    (Simplified version of their hybrid measure, on the MST only.)
    """
    from scipy.sparse.csgraph import minimum_spanning_tree
    N = C.shape[0]
    D = corr_distance(C) + 1e-9          # tiny offset: zero distances would be dropped as non-edges
    np.fill_diagonal(D, 0.0)
    mst = minimum_spanning_tree(D).tocoo()
    T = nx.Graph()
    T.add_nodes_from(range(N))
    T.add_weighted_edges_from(zip(mst.row.tolist(), mst.col.tolist(), mst.data.tolist()))
    deg = pd.Series(dict(T.degree(weight=None)))
    btw = pd.Series(nx.betweenness_centrality(T, weight="weight"))
    clo = pd.Series(nx.closeness_centrality(T, distance="weight"))
    ecc = pd.Series(nx.eccentricity(T))
    score = deg.rank() + btw.rank() + clo.rank() + (-ecc).rank()
    return [int(i) for i in score.sort_values().index[:k]]


def max_drawdown(x):
    """Maximum drawdown (negative number) of a return array or matrix (per column)."""
    w = np.cumprod(1.0 + np.asarray(x), axis=0)
    peak = np.maximum.accumulate(w, axis=0)
    return (w / peak - 1.0).min(axis=0)


def pca_kmeans_select(R_df, k, threshold=0.5, per_cluster=2, seed=0):
    """Re-implementation of the PCA-KMeans stratified selection (Vo, Wang & Nguyen, 2026).

    Features: 3-month return, Sharpe, 60-day volatility (stability), max drawdown,
    all computed over the last year of the look-back. Min-max scaled, risk
    features flipped, PC1 loadings (absolute, normalized) used as weights;
    stocks with score > threshold are clustered with K-Means and the top
    `per_cluster` by score are taken from each cluster.
    """
    X = R_df.values[-TRADING_DAYS:]
    ret3m = np.prod(1.0 + X[-63:], axis=0) - 1.0
    sd = X.std(axis=0, ddof=1) + 1e-12
    sharpe = X.mean(axis=0) / sd * np.sqrt(TRADING_DAYS)
    vol60 = X[-60:].std(axis=0, ddof=1)
    mdd = -max_drawdown(X)  # positive size of drawdown
    F = np.column_stack([ret3m, sharpe, vol60, mdd])
    mn, mx = F.min(axis=0), F.max(axis=0)
    F = (F - mn) / np.where(mx - mn > 1e-12, mx - mn, 1.0)
    F[:, 2] = 1.0 - F[:, 2]
    F[:, 3] = 1.0 - F[:, 3]
    w = np.abs(PCA(n_components=1).fit(F).components_[0])
    w = w / w.sum()
    score = F @ w
    good = np.where(score > threshold)[0]
    n_clusters = int(math.ceil(k / per_cluster))
    sel = []
    if len(good) > n_clusters:
        lab = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit_predict(F[good])
        for c in range(n_clusters):
            idx = good[lab == c]
            sel += [int(i) for i in idx[np.argsort(-score[idx])][:per_cluster]]
    else:
        sel = [int(i) for i in good]
    if len(sel) < k:
        rest = [int(i) for i in np.argsort(-score) if i not in sel]
        sel += rest[: k - len(sel)]
    return sel[:k]


### 2.5 Weighting schemes
Equal weights in the main experiments, so differences come from selection only. HRP is a full-universe benchmark.

In [ ]:
def hrp_weights(cov):
    """Hierarchical Risk Parity (Lopez de Prado, 2016) on a covariance matrix."""
    cov = np.asarray(cov)
    sd = np.sqrt(np.diag(cov))
    C = cov / np.outer(sd, sd)
    D = corr_distance(np.clip(C, -1, 1))
    np.fill_diagonal(D, 0.0)
    Z = linkage(squareform(D, checks=False), method="single")
    # quasi-diagonal order
    N = cov.shape[0]
    order = [int(Z[-1, 0]), int(Z[-1, 1])]
    while max(order) >= N:
        new = []
        for i in order:
            if i >= N:
                r = int(i - N)
                new += [int(Z[r, 0]), int(Z[r, 1])]
            else:
                new.append(i)
        order = new
    w = pd.Series(1.0, index=order)
    clusters = [order]
    while clusters:
        nxt = []
        for c in clusters:
            if len(c) <= 1:
                continue
            h = len(c) // 2
            c1, c2 = c[:h], c[h:]

            def cvar(ix):
                sub = cov[np.ix_(ix, ix)]
                iv = 1.0 / np.diag(sub)
                iv /= iv.sum()
                return float(iv @ sub @ iv)

            v1, v2 = cvar(c1), cvar(c2)
            alpha = 1.0 - v1 / (v1 + v2)
            w[c1] *= alpha
            w[c2] *= 1.0 - alpha
            nxt += [c1, c2]
        clusters = nxt
    w = w.sort_index().values
    return w / w.sum()


def minvar_weights(cov, max_w=1.0):
    """Long-only minimum-variance weights (fully invested), solved with SLSQP."""
    n = cov.shape[0]
    x0 = np.ones(n) / n
    res = minimize(lambda w: w @ cov @ w, x0, jac=lambda w: 2 * cov @ w,
                   bounds=[(0.0, max_w)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
                   method="SLSQP", options={"maxiter": 500, "ftol": 1e-12})
    w = np.clip(res.x if res.success else x0, 0, None)
    return w / w.sum()


def compute_weights(scheme, R_sel):
    """Portfolio weights for the selected assets, estimated on the look-back only."""
    n = R_sel.shape[1]
    if scheme == "ew" or n == 1:
        return np.ones(n) / n
    X = R_sel[-TRADING_DAYS:]
    if scheme == "invvol":
        iv = 1.0 / (X.std(axis=0, ddof=1) + 1e-12)
        return iv / iv.sum()
    cov = LedoitWolf().fit(X).covariance_
    if scheme == "minvar":
        return minvar_weights(cov)
    if scheme == "hrp":
        return hrp_weights(cov)
    raise ValueError(scheme)


### 2.6 Walk-forward backtest engine
Quarterly rebalancing, look-back data only, buy-and-hold between rebalances, costs on one-way turnover, out-of-sample diversification diagnostics per holding period.

In [ ]:
@dataclass
class Config:
    k: int = 10                  # number of assets to select
    lookback: int = 1260         # look-back in trading days (5 years)
    rebalance_every: int = 63    # trading days between rebalances (quarterly)
    stress_q: float = 0.10       # stress days = market in bottom 10% of the sub-window
    sub_len: int = 504           # sub-window length for robust aggregation (2 years)
    sub_step: int = 126          # step between sub-windows (6 months)
    agg_q: float = 0.75          # pessimistic quantile across sub-windows
    theta: float = 0.5           # quality-diversity trade-off
    kernel_a: float = 0.5        # affine kernel parameter
    tail_u: float = 0.10         # tail level for empirical tail dependence
    min_coverage: float = 0.95   # min share of non-missing days in the look-back
    cost_bps: float = 10.0       # one-way transaction cost (bps of traded value)
    test_start: str = "1990-01-01"
    test_end: str = None
    n_random: int = 20           # random-k replications


class Context:
    """Everything a selection method may use at one rebalance date (look-back data only).
    Expensive estimates are computed once and shared across methods."""

    def __init__(self, R_df, mkt, cfg, date=None):
        self.date = date
        self.R_df = R_df
        self.R = R_df.values
        self.mkt = np.asarray(mkt)
        self.cfg = cfg
        self.tickers = list(R_df.columns)

    @cached_property
    def z(self):
        return quality_scores(self.R_df).values

    @cached_property
    def C_plain(self):
        return shrunk_corr(self.R)

    @cached_property
    def C_robust(self):
        c = self.cfg
        return robust_stress_corr(self.R, self.mkt, c.stress_q, c.sub_len, c.sub_step, c.agg_q)[0]

    @cached_property
    def C_stress_single(self):
        c = self.cfg
        n = len(self.mkt)
        return robust_stress_corr(self.R, self.mkt, c.stress_q, n, n, 0.5)[0]

    @cached_property
    def C_plain_robust(self):
        """Robust aggregation across sub-windows but on ALL days (no stress conditioning)."""
        c = self.cfg
        return robust_stress_corr(self.R, self.mkt, 1.0, c.sub_len, c.sub_step, c.agg_q)[0]

    @cached_property
    def taildep(self):
        return lower_tail_dependence(self.R, self.cfg.tail_u)


# Each selection method: f(ctx) -> list of column indices.
def m_srdas(ctx, theta=None):
    th = ctx.cfg.theta if theta is None else theta
    return logdet_select(ctx.C_robust, ctx.z, ctx.cfg.k, th, ctx.cfg.kernel_a)


def m_srdas_noq(ctx):
    return logdet_select(ctx.C_robust, ctx.z, ctx.cfg.k, 0.0, ctx.cfg.kernel_a)


def m_das_plain(ctx):
    return logdet_select(ctx.C_plain, ctx.z, ctx.cfg.k, ctx.cfg.theta, ctx.cfg.kernel_a)


def m_das_plain_robust(ctx):
    return logdet_select(ctx.C_plain_robust, ctx.z, ctx.cfg.k, ctx.cfg.theta, ctx.cfg.kernel_a)


def m_das_stress_single(ctx):
    return logdet_select(ctx.C_stress_single, ctx.z, ctx.cfg.k, ctx.cfg.theta, ctx.cfg.kernel_a)


def m_stress_cluster(ctx):
    return cluster_pick(corr_distance(ctx.C_robust), ctx.z, ctx.cfg.k)


def m_corr_cluster(ctx):
    return cluster_pick(corr_distance(ctx.C_plain), ctx.z, ctx.cfg.k)


def m_taildep_cluster(ctx):
    return cluster_pick(1.0 - ctx.taildep, ctx.z, ctx.cfg.k)


def m_mst_periphery(ctx):
    return mst_periphery_select(ctx.C_plain, ctx.cfg.k)


def m_pca_kmeans(ctx):
    return pca_kmeans_select(ctx.R_df, ctx.cfg.k)


def m_topk_quality(ctx):
    return [int(i) for i in np.argsort(-ctx.z)[: ctx.cfg.k]]


def make_random(seed):
    def m_random(ctx):
        day = ctx.date.toordinal() if ctx.date is not None else 0
        rng = np.random.default_rng([seed, day])
        return [int(i) for i in rng.choice(len(ctx.tickers), size=min(ctx.cfg.k, len(ctx.tickers)), replace=False)]
    return m_random


def m_all(ctx):
    return list(range(len(ctx.tickers)))


@dataclass
class Strategy:
    name: str
    select: callable
    weights: str = "ew"
    group: str = ""


def default_strategies(cfg, weights="ew", include_random=True):
    """The full comparison set: proposed method, ablations, baselines, benchmarks."""
    S = [
        Strategy("SR-DAS", m_srdas, weights, "Proposed"),
        # ablations (each removes one component of SR-DAS)
        Strategy("Abl: Stress-Cluster (rule)", m_stress_cluster, weights, "Ablation"),
        Strategy("Abl: DAS all-days (stress)", m_das_plain_robust, weights, "Ablation"),
        Strategy("Abl: DAS stress 1-window (robust)", m_das_stress_single, weights, "Ablation"),
        Strategy("Abl: SR-DAS theta=0 (quality)", m_srdas_noq, weights, "Ablation"),
        Strategy("Abl: DAS plain corr (all)", m_das_plain, weights, "Ablation"),
        # literature baselines
        Strategy("TailDep-Cluster", m_taildep_cluster, weights, "Baseline"),
        Strategy("Corr-Cluster", m_corr_cluster, weights, "Baseline"),
        Strategy("MST-Periphery", m_mst_periphery, weights, "Baseline"),
        Strategy("PCA-KMeans (Vo et al.)", m_pca_kmeans, weights, "Baseline"),
        Strategy("Top-k Quality", m_topk_quality, weights, "Baseline"),
        # full-universe benchmarks (not selection methods)
        Strategy("EW-All (1/N)", m_all, "ew", "Benchmark"),
        Strategy("HRP-All", m_all, "hrp", "Benchmark"),
    ]
    if include_random:
        S += [Strategy(f"Random-k #{s}", make_random(s), weights, "Random") for s in range(cfg.n_random)]
    return S


@dataclass
class Result:
    name: str
    group: str
    returns: pd.Series                      # daily net portfolio returns
    gross_returns: pd.Series
    turnover: pd.Series                     # one-way turnover at each rebalance
    holdings: dict = field(default_factory=dict)   # rebalance date -> {ticker: weight}
    diag: pd.DataFrame = None               # per-period diagnostics


def drifted_weights(w, R_hold):
    """Buy-and-hold weights at the end of a holding period."""
    v = w * np.prod(1.0 + R_hold, axis=0)
    s = v.sum()
    return v / s if s > 0 else w


def avg_pairwise_corr(X):
    sd = X.std(axis=0, ddof=1)
    ok = sd > 1e-12
    if ok.sum() < 2 or X.shape[0] < 5:
        return np.nan
    C = np.corrcoef(X[:, ok], rowvar=False)
    return (C.sum() - np.trace(C)) / (C.shape[0] * (C.shape[0] - 1))


def period_diagnostics(R_hold, w):
    """Out-of-sample diversification of the held basket over one holding period."""
    n = R_hold.shape[1]
    if n < 2 or R_hold.shape[0] < 5:
        return np.nan, np.nan
    sd = R_hold.std(axis=0, ddof=1)
    ok = sd > 1e-12
    if ok.sum() < 2:
        return np.nan, np.nan
    C = np.corrcoef(R_hold[:, ok], rowvar=False)
    avg_corr = (C.sum() - np.trace(C)) / (C.shape[0] * (C.shape[0] - 1))
    port_sd = (R_hold @ w).std(ddof=1)
    dr = float(w @ sd / port_sd) if port_sd > 1e-12 else np.nan
    return avg_corr, dr


def run_backtest(returns, market, strategies, cfg, universe_fn=None, verbose=True, eval_stress_q=0.10):
    """Walk-forward backtest.

    At each rebalance date t (every cfg.rebalance_every days):
      * eligible assets = universe_fn(t) with >= min_coverage data in the look-back
        and a valid return on day t;
      * each strategy selects/weights using ONLY data in (t-lookback, t];
      * portfolios are held buy-and-hold from t+1 to the next rebalance.
    Missing returns while held (e.g. delisting) are treated as 0 (cash).
    Transaction cost = cost_bps * one-way turnover, charged on the first day.
    """
    returns = returns.sort_index()
    market = market.reindex(returns.index)
    dates = returns.index
    t0 = dates.searchsorted(pd.Timestamp(cfg.test_start))
    t0 = max(t0, cfg.lookback)
    t_end = len(dates) if cfg.test_end is None else dates.searchsorted(pd.Timestamp(cfg.test_end), side="right")
    rebal_idx = list(range(t0 - 1, t_end - 1, cfg.rebalance_every))
    # Evaluation-only threshold for "stress days" in the test period (never used for selection)
    eval_thr = market.iloc[t0:t_end].quantile(eval_stress_q)

    out = {s.name: dict(net=[], gross=[], turn={}, hold={}, diag=[], prev=None) for s in strategies}
    tic = time.time()
    for n_r, t in enumerate(rebal_idx):
        t_next = min(t + cfg.rebalance_every, t_end - 1)
        if t_next <= t:
            break
        win = returns.iloc[t - cfg.lookback + 1: t + 1]
        mkt_win = market.iloc[t - cfg.lookback + 1: t + 1]
        cols = list(win.columns) if universe_fn is None else [c for c in universe_fn(dates[t]) if c in win.columns]
        w_df = win[cols]
        cov_ok = w_df.notna().mean() >= cfg.min_coverage
        last_ok = w_df.iloc[-1].notna()
        elig = [c for c in cols if cov_ok[c] and last_ok[c]]
        mk_ok = mkt_win.notna().values
        R_df = w_df.loc[mk_ok, elig].fillna(0.0)
        ctx = Context(R_df, mkt_win.values[mk_ok], cfg, dates[t])
        hold = returns.iloc[t + 1: t_next + 1]
        hold_stress = (market.iloc[t + 1: t_next + 1] <= eval_thr).values
        for s in strategies:
            o = out[s.name]
            idx = s.select(ctx)
            tick = [elig[i] for i in idx]
            w = compute_weights(s.weights, R_df.values[:, idx])
            R_hold = hold[tick].fillna(0.0).values
            # buy-and-hold path
            growth = np.cumprod(1.0 + R_hold, axis=0) * w
            val = np.concatenate([[1.0], growth.sum(axis=1)])
            r_gross = val[1:] / val[:-1] - 1.0
            # turnover vs drifted previous portfolio
            new = pd.Series(w, index=tick)
            prev = o["prev"]
            if prev is None:
                turn = 1.0
            else:
                u = new.index.union(prev.index)
                turn = 0.5 * float(np.abs(new.reindex(u, fill_value=0) - prev.reindex(u, fill_value=0)).sum())
            r_net = r_gross.copy()
            r_net[0] -= turn * cfg.cost_bps / 1e4
            o["gross"].append(pd.Series(r_gross, index=hold.index))
            o["net"].append(pd.Series(r_net, index=hold.index))
            o["turn"][dates[t]] = turn
            o["hold"][dates[t]] = dict(zip(tick, np.round(w, 6)))
            o["prev"] = pd.Series(drifted_weights(w, R_hold), index=tick)
            ac, dr = period_diagnostics(R_hold, w)
            n_st = int(hold_stress.sum())
            ac_st = avg_pairwise_corr(R_hold[hold_stress]) if n_st >= 5 else np.nan
            o["diag"].append(dict(date=dates[t], n_eligible=len(elig), n_held=len(tick),
                                  avg_corr_oos=ac, div_ratio_oos=dr,
                                  stress_corr_oos=ac_st, n_stress_days=n_st))
        if verbose and (n_r % 10 == 0 or n_r == len(rebal_idx) - 1):
            print(f"  rebalance {n_r + 1}/{len(rebal_idx)}  {dates[t].date()}  eligible={len(elig)}  "
                  f"elapsed={time.time() - tic:.0f}s")
    results = {}
    for s in strategies:
        o = out[s.name]
        results[s.name] = Result(s.name, s.group, pd.concat(o["net"]), pd.concat(o["gross"]),
                                 pd.Series(o["turn"]), o["hold"], pd.DataFrame(o["diag"]).set_index("date"))
    return results


### 2.7 Metrics and Sharpe tests

In [ ]:
def perf_metrics(r, rf=None):
    """Standard performance metrics of a daily return series."""
    r = r.dropna()
    rf_ = pd.Series(0.0, index=r.index) if rf is None else rf.reindex(r.index).fillna(0.0)
    ex = r - rf_
    yrs = len(r) / TRADING_DAYS
    wealth = (1 + r).cumprod()
    cagr = wealth.iloc[-1] ** (1 / yrs) - 1 if yrs > 0 else np.nan
    vol = r.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = ex.mean() / ex.std(ddof=1) * np.sqrt(TRADING_DAYS)
    dn = ex[ex < 0]
    sortino = ex.mean() * TRADING_DAYS / (np.sqrt((dn ** 2).sum() / len(ex)) * np.sqrt(TRADING_DAYS))
    mdd = (wealth / wealth.cummax() - 1).min()
    cvar = r[r <= r.quantile(0.05)].mean()
    return dict(CAGR=cagr, Vol=vol, Sharpe=sharpe, Sortino=sortino, MaxDD=mdd,
                Calmar=cagr / abs(mdd) if mdd < 0 else np.nan, CVaR5_daily=cvar)


def stress_stats(r, market, q=0.10):
    """Out-of-sample behaviour on market stress days (bottom q of the market in the window)."""
    df = pd.concat([r, market], axis=1, keys=["p", "m"], sort=True).dropna()
    st = df[df["m"] <= df["m"].quantile(q)]
    beta = np.cov(st["p"], st["m"])[0, 1] / st["m"].var() if len(st) > 5 else np.nan
    return dict(StressBeta=beta, StressRet_daily=st["p"].mean())


def summary_table(results, rf=None, start=None, end=None, collapse_random=True, market=None):
    """One row per strategy; Random-k replications are averaged into one row.

    If `market` is given, adds out-of-sample stress diagnostics: beta and mean return on
    market stress days, and the average pairwise correlation of the held basket on
    stress days (weighted by number of stress days per holding period)."""
    rows = {}
    for name, res in results.items():
        r = res.returns.loc[start:end]
        if len(r) < 20:
            continue
        m = perf_metrics(r, rf)
        yrs = len(r) / TRADING_DAYS
        tv = res.turnover.loc[start:end] if start or end else res.turnover
        m["Turnover/yr"] = tv.sum() / yrs if yrs > 0 else np.nan
        d = res.diag.loc[start:end] if res.diag is not None else None
        m["AvgCorr_OOS"] = d["avg_corr_oos"].mean() if d is not None else np.nan
        m["DivRatio_OOS"] = d["div_ratio_oos"].mean() if d is not None else np.nan
        if d is not None and "stress_corr_oos" in d:
            ok = d["stress_corr_oos"].notna()
            m["StressCorr_OOS"] = np.average(d.loc[ok, "stress_corr_oos"], weights=d.loc[ok, "n_stress_days"]) if ok.any() else np.nan
        if market is not None:
            m.update(stress_stats(r, market.loc[start:end]))
        m["Group"] = res.group
        rows[name] = m
    df = pd.DataFrame(rows).T
    if collapse_random:
        rnd = [n for n in df.index if n.startswith("Random-k #")]
        if rnd:
            num = df.loc[rnd].drop(columns="Group").astype(float).mean()
            num["Group"] = "Random"
            df = df.drop(index=rnd)
            df.loc[f"Random-k (mean of {len(rnd)})"] = num
    order = {"Proposed": 0, "Proposed-2": 0.5, "Ablation": 1, "Original": 1.5, "Baseline": 2, "Random": 3, "Benchmark": 4}
    df["_o"] = df["Group"].map(order)
    df = df.sort_values(["_o"], kind="stable").drop(columns="_o")
    cols = ["Group", "CAGR", "Vol", "Sharpe", "Sortino", "MaxDD", "Calmar", "CVaR5_daily",
            "Turnover/yr", "AvgCorr_OOS", "StressCorr_OOS", "DivRatio_OOS", "StressBeta", "StressRet_daily"]
    return df[[c for c in cols if c in df.columns]]


def format_table(df):
    fmt = {"CAGR": "{:.2%}", "Vol": "{:.2%}", "Sharpe": "{:.3f}", "Sortino": "{:.3f}",
           "MaxDD": "{:.2%}", "Calmar": "{:.3f}", "CVaR5_daily": "{:.2%}", "Turnover/yr": "{:.2f}",
           "AvgCorr_OOS": "{:.3f}", "DivRatio_OOS": "{:.3f}", "StressCorr_OOS": "{:.3f}",
           "StressBeta": "{:.3f}", "StressRet_daily": "{:.2%}"}
    out = df.copy()
    for c, f in fmt.items():
        if c in out:
            out[c] = out[c].map(lambda v: f.format(v) if pd.notna(v) else "")
    return out


def crisis_table(results, windows, market=None):
    """Cumulative return and max drawdown of each strategy inside crisis windows."""
    rows = []
    series = {n: r.returns for n, r in results.items() if not n.startswith("Random-k #")}
    rnd = [r.returns for n, r in results.items() if n.startswith("Random-k #")]
    if rnd:
        series[f"Random-k (mean of {len(rnd)})"] = pd.concat(rnd, axis=1, sort=True).mean(axis=1)
    if market is not None:
        series["Market"] = market
    for label, (a, b) in windows.items():
        for n, r in series.items():
            x = r.loc[a:b].dropna()
            if len(x) < 5:
                continue
            w = (1 + x).cumprod()
            rows.append(dict(Crisis=label, Strategy=n, Return=w.iloc[-1] - 1,
                             MaxDD=(w / w.cummax() - 1).min()))
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    return df.pivot(index="Strategy", columns="Crisis", values="MaxDD"), \
        df.pivot(index="Strategy", columns="Crisis", values="Return")


def _sharpe(x):
    s = x.std(axis=0, ddof=1)
    return x.mean(axis=0) / np.where(s > 0, s, np.nan)


def sharpe_test(a, b, rf=None, block=21, B=2000, seed=0):
    """Test H0: Sharpe(a) = Sharpe(b) for two daily return series.

    * Jobson-Korkie test with Memmel (2003) correction (assumes iid normal).
    * Circular block bootstrap (block length `block`) of the Sharpe difference,
      centred under H0 — robust to autocorrelation and fat tails, in the
      spirit of Ledoit & Wolf (2008).
    Returns annualized difference and both p-values.
    """
    df = pd.concat([a, b], axis=1, sort=True).dropna()
    if rf is not None:
        r = rf.reindex(df.index).fillna(0.0).values[:, None]
        X = df.values - r
    else:
        X = df.values
    T = len(X)
    sr = _sharpe(X)
    diff = (sr[0] - sr[1]) * np.sqrt(TRADING_DAYS)
    rho = np.corrcoef(X.T)[0, 1]
    var_jk = (2 * (1 - rho) + 0.5 * (sr[0] ** 2 + sr[1] ** 2 - 2 * sr[0] * sr[1] * rho ** 2)) / T
    p_jk = 2 * (1 - norm.cdf(abs(sr[0] - sr[1]) / np.sqrt(var_jk)))
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    d_obs = sr[0] - sr[1]
    boots = np.empty(B)
    for i in range(B):
        st = rng.integers(0, T, nb)
        idx = ((st[:, None] + np.arange(block)[None, :]) % T).ravel()[:T]
        s = _sharpe(X[idx])
        boots[i] = s[0] - s[1]
    p_bs = float(np.mean(np.abs(boots - d_obs) >= abs(d_obs)))
    return dict(SharpeDiff_ann=diff, p_JK_Memmel=p_jk, p_block_bootstrap=p_bs)


def sharpe_tests_vs(results, ref, rf=None, B=2000):
    rows = {}
    for n, r in results.items():
        if n == ref or n.startswith("Random-k #"):
            continue
        rows[n] = sharpe_test(results[ref].returns, r.returns, rf, B=B)
    return pd.DataFrame(rows).T


def selection_frequency(result):
    """How often each asset was held across rebalances."""
    cnt = {}
    for h in result.holdings.values():
        for t in h:
            cnt[t] = cnt.get(t, 0) + 1
    return pd.Series(cnt).sort_values(ascending=False) / max(1, len(result.holdings))


### 2.8 Plotting helpers

In [ ]:
import matplotlib.pyplot as plt

PALETTE = {"Proposed": "#1f4e9c", "Proposed-2": "#2a9d8f", "Original": "#e9a23b", "Ablation": "#8fb0de", "Baseline": "#d1495b",
           "Random": "#9a9a9a", "Benchmark": "#2b2b2b"}


def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)


def plot_wealth(results, names, title="Cumulative wealth (log scale, net of costs)"):
    fig, ax = plt.subplots(figsize=(11, 5))
    for n in names:
        r = results[n]
        w = (1 + r.returns).cumprod()
        lw = 2.4 if r.group == "Proposed" else 1.2
        ax.plot(w.index, w.values, lw=lw, label=n,
                color=PALETTE.get(r.group) if r.group in ("Proposed", "Benchmark") else None)
    ax.set_yscale("log"); ax.set_title(title, loc="left"); ax.legend(fontsize=8, ncol=2, frameon=False)
    _style(ax); plt.tight_layout(); plt.show()


def plot_drawdowns(results, names, title="Drawdowns"):
    fig, ax = plt.subplots(figsize=(11, 4))
    for n in names:
        r = results[n]
        w = (1 + r.returns).cumprod()
        ax.plot(w.index, (w / w.cummax() - 1).values, lw=2 if r.group == "Proposed" else 1, label=n)
    ax.set_title(title, loc="left"); ax.legend(fontsize=8, ncol=2, frameon=False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    _style(ax); plt.tight_layout(); plt.show()


def plot_metric_bars(table, metric, title=None, higher_is_better=True):
    t = table[metric].astype(float).sort_values(ascending=not higher_is_better)
    colors = [PALETTE.get(g, "#888") for g in table.loc[t.index, "Group"]]
    fig, ax = plt.subplots(figsize=(9, 0.35 * len(t) + 1))
    ax.barh(t.index[::-1], t.values[::-1], color=colors[::-1])
    ax.set_title(title or metric, loc="left"); _style(ax); ax.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.show()


def plot_sensitivity(sens, metrics=("Sharpe", "MaxDD", "StressCorr_OOS")):
    params = sens["param"].unique()
    fig, axes = plt.subplots(len(metrics), len(params), figsize=(3.2 * len(params), 2.6 * len(metrics)), squeeze=False)
    for j, p in enumerate(params):
        d = sens[sens["param"] == p]
        for i, m in enumerate(metrics):
            ax = axes[i, j]
            ax.plot(d["value"].astype(str), d[m].astype(float), marker="o", color=PALETTE["Proposed"])
            if i == 0: ax.set_title(p)
            if j == 0: ax.set_ylabel(m)
            _style(ax)
    plt.tight_layout(); plt.show()


### 2.9 Revised methods and hypothesis tests (new in this notebook)

* **`balanced_theta`** — the fix for the scale problem. The objective is `2θ Σ z_i + log det K_S`. At each rebalance θ is set so that the cross-sectional spread of the quality term equals the spread of each asset's diversity contribution (`−log (K⁻¹)_ii`, the log-det "uniqueness" of the asset). A fixed rule; no return data is used to choose it.
* **`make_screen`** — screen-then-diversify: top 50% by quality, then pure log-det diversity.
* **`paired_period_test`** — Wilcoxon signed-rank test across holding periods on the held basket's out-of-sample stress-day correlation.
* **`ratio_ci`** — block-bootstrap 95% CI for volatility ratios.
* **`holm`** — Holm–Bonferroni correction for the many Sharpe comparisons.
* **`evaluate_hypotheses`** — evaluates H1–H7 exactly as pre-registered.

In [ ]:
from scipy.linalg import pinvh
from scipy.stats import wilcoxon

THETA_LOG = {}   # rebalance date -> balanced theta chosen by the primary method (for reporting)


def uniqueness_scores(K):
    """Diversity contribution of each asset: log of its conditional variance given all
    other assets under kernel K, i.e. log det(K) - log det(K without i) = -log (K^-1)_ii.
    These are the canonical per-asset diversity scores of the log-det objective."""
    Kinv = pinvh(K)
    return -np.log(np.clip(np.diag(Kinv), 1e-12, None))


def balanced_theta(K, z, beta=1.0):
    """Scale rule that fixes the quality/diversity imbalance of the original SR-DAS.

    The objective is  2*theta*sum(z_S) + log det K_S. theta is set at every rebalance so
    that the cross-sectional spread of the quality term (2*theta*z_i) equals the spread
    of the diversity scores (uniqueness_scores). beta = 1 gives both terms equal weight.
    This is a fixed rule — nothing is tuned on returns."""
    g = uniqueness_scores(K)
    return beta * np.std(g) / (2.0 * np.std(z) + 1e-12)


def make_balanced(corr_attr, log=False):
    """Balanced SR-DAS on a given dependence estimate (attribute name of Context)."""
    def m(ctx):
        C = getattr(ctx, corr_attr)
        K = affine_corr_kernel(C, ctx.cfg.kernel_a)
        th = balanced_theta(K, ctx.z)
        if log and ctx.date is not None:
            THETA_LOG[ctx.date] = th
        return logdet_select(C, ctx.z, ctx.cfg.k, th, ctx.cfg.kernel_a)
    return m


def make_screen(corr_attr, frac=0.5):
    """Screen-then-diversify: keep the top `frac` of assets by quality, then pure
    log-det diversity (theta = 0) among them on the given dependence estimate."""
    def m(ctx):
        C = getattr(ctx, corr_attr)
        n = len(ctx.z)
        keep = np.argsort(-ctx.z)[: max(ctx.cfg.k, int(np.ceil(frac * n)))]
        keep = np.sort(keep)
        sub = logdet_select(C[np.ix_(keep, keep)], ctx.z[keep], ctx.cfg.k, 0.0, ctx.cfg.kernel_a)
        return [int(keep[i]) for i in sub]
    return m


def make_theta(theta):
    """SR-DAS with a fixed theta (used for the trade-off frontier)."""
    def m(ctx):
        return logdet_select(ctx.C_robust, ctx.z, ctx.cfg.k, theta, ctx.cfg.kernel_a)
    return m


def v3_strategies(cfg, include_random=True, weights="ew"):
    """Pre-specified comparison set for the revised study."""
    S = [
        Strategy("B-SR-DAS (balanced)", make_balanced("C_robust", log=True), weights, "Proposed"),
        Strategy("S-SR-DAS (screen+diversify)", make_screen("C_robust"), weights, "Proposed-2"),
        # ablations of the PRIMARY method: one ingredient changed each
        Strategy("Abl: B-DAS all-days (stress)", make_balanced("C_plain_robust"), weights, "Ablation"),
        Strategy("Abl: B-DAS stress 1-window (robust)", make_balanced("C_stress_single"), weights, "Ablation"),
        Strategy("Abl: Stress-Cluster (rule)", m_stress_cluster, weights, "Ablation"),
        # the original specification, for reference
        Strategy("Original SR-DAS (theta=0.5)", m_srdas, weights, "Original"),
        Strategy("Original SR-DAS (theta=0)", m_srdas_noq, weights, "Original"),
        # literature baselines and benchmarks
        Strategy("TailDep-Cluster", m_taildep_cluster, weights, "Baseline"),
        Strategy("Corr-Cluster", m_corr_cluster, weights, "Baseline"),
        Strategy("MST-Periphery", m_mst_periphery, weights, "Baseline"),
        Strategy("PCA-KMeans (Vo et al.)", m_pca_kmeans, weights, "Baseline"),
        Strategy("Top-k Quality", m_topk_quality, weights, "Baseline"),
        Strategy("EW-All (1/N)", m_all, "ew", "Benchmark"),
        Strategy("HRP-All", m_all, "hrp", "Benchmark"),
    ]
    if include_random:
        S += [Strategy(f"Random-k #{s}", make_random(s), weights, "Random") for s in range(cfg.n_random)]
    return S


# ---------- statistical helpers for the pre-specified hypotheses ----------
def paired_period_test(res_a, res_b, col="stress_corr_oos", alternative="less"):
    """Paired test across holding periods of a per-period diagnostic (e.g. the held
    basket's stress-day correlation). Wilcoxon signed-rank on period-by-period
    differences a - b. alternative='less' tests whether a is systematically lower."""
    d = (res_a.diag[col] - res_b.diag[col]).dropna()
    if len(d) < 6 or np.allclose(d, 0):
        return dict(n_periods=len(d), mean_diff=float(d.mean()) if len(d) else np.nan,
                    share_a_lower=float((d < 0).mean()) if len(d) else np.nan, p_value=np.nan)
    p = wilcoxon(d, alternative=alternative).pvalue
    return dict(n_periods=len(d), mean_diff=float(d.mean()), share_a_lower=float((d < 0).mean()), p_value=float(p))


def ratio_ci(a, b, stat="vol", block=21, B=2000, seed=0, q=0.10, market=None):
    """Block-bootstrap 95% CI for a risk ratio of two daily return series:
    stat='vol' -> vol(a)/vol(b);  stat='cvar' -> CVaR5(a)/CVaR5(b)."""
    X = pd.concat([a, b], axis=1, sort=True).dropna().values
    T = len(X)

    def f(Y):
        if stat == "vol":
            s = Y.std(axis=0, ddof=1)
            return s[0] / s[1]
        lo = [Y[Y[:, j] <= np.quantile(Y[:, j], 0.05), j].mean() for j in range(2)]
        return lo[0] / lo[1]

    obs = f(X)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    bs = np.empty(B)
    for i in range(B):
        idx = ((rng.integers(0, T, nb)[:, None] + np.arange(block)[None, :]) % T).ravel()[:T]
        bs[i] = f(X[idx])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return dict(ratio=float(obs), ci_low=float(lo), ci_high=float(hi))


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (controls family-wise error)."""
    p = pd.Series(pvals, dtype=float)
    ok = p.dropna().sort_values()
    m = len(ok)
    adj = pd.Series(np.nan, index=p.index)
    running = 0.0
    for i, (k, v) in enumerate(ok.items()):
        running = max(running, min(1.0, (m - i) * v))
        adj[k] = running
    return adj


def sharpe_tests_holm(results, ref, rf=None, B=2000):
    t = sharpe_tests_vs(results, ref, rf, B=B)
    t["p_holm (bootstrap)"] = holm(t["p_block_bootstrap"])
    return t


def evaluate_hypotheses(res, primary, rf, B=2000, core_only=False):
    """Evaluate the pre-registered hypotheses for one method on one sample.
    H1-H4 apply to any method; H5-H7 (ablations) apply to the primary method only
    (core_only=True skips them). Returns: hypothesis, statistic, p-value / CI, verdict."""
    rows = []

    def add(h, desc, stat, detail, ok):
        rows.append(dict(Hypothesis=h, Description=desc, Statistic=stat, Detail=detail,
                         Verdict="SUPPORTED" if ok else "NOT supported"))

    P = res[primary]
    # H1: lower stress-day correlation of the held basket than Top-k Quality
    t = paired_period_test(P, res["Top-k Quality"])
    add("H1", "Stress-day basket correlation lower than Top-k Quality (paired, one-sided)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    # H2: lower volatility than 1/N
    c = ratio_ci(P.returns, res["EW-All (1/N)"].returns, "vol", B=B)
    add("H2", "Volatility lower than EW-All (1/N)", f"ratio {c['ratio']:.3f}",
        f"95% CI [{c['ci_low']:.3f}, {c['ci_high']:.3f}]", c["ci_high"] < 1)
    # H3: lower volatility than pure diversity (theta = 0) -> the balance works
    c = ratio_ci(P.returns, res["Original SR-DAS (theta=0)"].returns, "vol", B=B)
    add("H3", "Volatility lower than pure-diversity SR-DAS (theta=0)", f"ratio {c['ratio']:.3f}",
        f"95% CI [{c['ci_low']:.3f}, {c['ci_high']:.3f}]", c["ci_high"] < 1)
    # H4: no significant Sharpe under-performance vs 1/N
    s = sharpe_test(P.returns, res["EW-All (1/N)"].returns, rf, B=B)
    add("H4", "Sharpe not significantly below EW-All (1/N) (two-sided, 5%)",
        f"diff {s['SharpeDiff_ann']:+.3f}", f"p = {s['p_block_bootstrap']:.4f}",
        not (s["SharpeDiff_ann"] < 0 and s["p_block_bootstrap"] < 0.05))
    if core_only:
        return pd.DataFrame(rows).set_index("Hypothesis")
    # H5: lower stress-day correlation than cluster-then-pick on the same matrix (RQ1)
    t = paired_period_test(P, res["Abl: Stress-Cluster (rule)"])
    add("H5", "Stress-day basket correlation lower than Stress-Cluster rule (RQ1, one-sided)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    # H6: stress conditioning lowers realized stress correlation (RQ2)
    t = paired_period_test(P, res["Abl: B-DAS all-days (stress)"])
    add("H6", "Stress conditioning lowers stress-day basket correlation vs all-day correlation (RQ2)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    # H7: robust aggregation lowers realized stress correlation (RQ3)
    t = paired_period_test(P, res["Abl: B-DAS stress 1-window (robust)"])
    add("H7", "Robust aggregation lowers stress-day basket correlation vs single window (RQ3)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    return pd.DataFrame(rows).set_index("Hypothesis")


def run_frontier(returns, market, cfg, thetas, rf=None):
    """SR-DAS for a grid of fixed theta values: maps the quality-diversity trade-off."""
    S = [Strategy(f"theta={t:g}", make_theta(t), "ew", "Proposed") for t in thetas]
    r = run_backtest(returns, market, S, cfg, verbose=False)
    tab = summary_table(r, rf, market=market)
    tab["theta"] = thetas
    return tab


def plot_frontier(frontier, points, x="StressCorr_OOS", y="Vol", title=None):
    """Frontier of fixed-theta SR-DAS (line) with other strategies as labelled points."""
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    f = frontier.sort_values("theta")
    ax.plot(f[x].astype(float), f[y].astype(float), "-o", color=PALETTE["Proposed"], lw=1.5, ms=4,
            label="SR-DAS, fixed theta")
    for _, r in f.iterrows():
        ax.annotate(f"θ={r['theta']:g}", (float(r[x]), float(r[y])), fontsize=7, xytext=(4, 2),
                    textcoords="offset points", color=PALETTE["Proposed"])
    for name, r in points.iterrows():
        g = r["Group"]
        ax.scatter(float(r[x]), float(r[y]), s=60 if g in ("Proposed", "Proposed-2") else 30,
                   color=PALETTE.get(g, "#888"), zorder=3,
                   marker="*" if g in ("Proposed", "Proposed-2") else "o")
        ax.annotate(name, (float(r[x]), float(r[y])), fontsize=7, xytext=(4, -9), textcoords="offset points")
    ax.set_xlabel(f"{x} (lower = more diversified in stress)")
    ax.set_ylabel(f"{y} (lower = better)")
    if y in ("Vol",):
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title or f"Trade-off: {x} vs {y}", loc="left")
    _style(ax); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


## 3. Data

Daily value-weighted returns of the 49 Fama–French industries and the daily Fama–French factors (market return and T-bill rate), downloaded from the Kenneth French Data Library and cached in `data/`. A few industries have no data in the 1960s; the backtest simply treats an industry as eligible once it has 95% of its look-back history.

In [ ]:
import io
import os
import zipfile
import requests

FRENCH_BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"


def _download_zip_text(filename, cache_dir="data"):
    """Download a Kenneth French zip file (cached locally) and return the CSV text."""
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, filename)
    if not os.path.exists(path):
        url = FRENCH_BASE + filename
        print("Downloading", url)
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (research script)"}, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    with zipfile.ZipFile(path) as z:
        name = [n for n in z.namelist() if n.lower().endswith((".csv", ".txt"))][0]
        return z.read(name).decode("latin-1")


def parse_french_csv(text, block=0):
    """Parse a Kenneth French CSV file and return block `block` as a DataFrame.

    The files contain one or more tables (e.g. value-weighted, then equal-weighted
    returns). Each table starts with a header line beginning with ',' and continues
    with rows whose first field is a YYYYMMDD date. Values are in percent and
    missing values are coded -99.99 or -999; both are converted to NaN and
    returns are converted to decimals.
    """
    lines = text.splitlines()
    blocks, cur, header = [], None, None
    for line in lines:
        parts = [p.strip() for p in line.split(",")]
        if line.startswith(",") and len(parts) > 2:
            header = parts[1:]
            cur = []
            blocks.append((header, cur))
            continue
        if cur is not None:
            if len(parts) >= 2 and parts[0].isdigit() and len(parts[0]) == 8:
                cur.append(parts[: len(header) + 1])
            elif cur:  # end of the current table
                cur = None
    header, rows = blocks[block]
    df = pd.DataFrame(rows, columns=["date"] + header)
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    df = df.set_index("date").apply(pd.to_numeric, errors="coerce")
    df = df.mask(df <= -99.99)
    return df / 100.0


def load_french_industries(n=49, start="1985-01-01", cache_dir="data", weighting="value"):
    """Daily returns of the Fama-French n-industry portfolios (decimal)."""
    text = _download_zip_text(f"{n}_Industry_Portfolios_daily_CSV.zip", cache_dir)
    df = parse_french_csv(text, block=0 if weighting == "value" else 1)
    return df.loc[start:]


def load_french_factors(start="1985-01-01", cache_dir="data"):
    """Daily Fama-French factors: returns DataFrame with Mkt (total market return), Mkt-RF, RF."""
    text = _download_zip_text("F-F_Research_Data_Factors_daily_CSV.zip", cache_dir)
    f = parse_french_csv(text, block=0)
    f["Mkt"] = f["Mkt-RF"] + f["RF"]
    return f.loc[start:]


In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

def save_table(df, name):
    df.to_csv(os.path.join(OUT_DIR, name + ".csv"))
    if SAVE_TO_DRIVE:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        df.to_csv(os.path.join(DRIVE_DIR, name + ".csv"))

with open(os.path.join(OUT_DIR, "specification.json"), "w") as f:
    f.write(SPEC_JSON + "\n\nfingerprint: " + SPEC_HASH + "\n")

ind = load_french_industries(49, start="1963-07-01", cache_dir=DATA_DIR)
ff = load_french_factors(start="1963-07-01", cache_dir=DATA_DIR)
common = ind.index.intersection(ff.index)
ind, ff = ind.loc[common], ff.loc[common]
MKT, RF = ff["Mkt"], ff["RF"]
print(f"{ind.shape[1]} industries, {ind.index[0].date()} to {ind.index[-1].date()}")
avail = ind.notna().groupby(ind.index.year).mean().mean(axis=1) * ind.shape[1]
print("Average number of industries with data, selected years:")
display(avail.loc[[1964, 1966, 1969, 1970, 1975, 1980, 1990]].round(1).to_frame("industries"))

In [ ]:
CRISES_HOLDOUT = {
    "1973-74 bear market": ("1973-01-11", "1974-10-03"),
    "1980 Q1 sell-off": ("1980-02-13", "1980-03-27"),
    "1981-82 recession": ("1980-11-28", "1982-08-12"),
    "1987 crash": ("1987-08-25", "1987-12-04"),
}
CRISES_REPLICATION = {
    "Gulf war 1990": ("1990-07-16", "1990-10-11"),
    "LTCM 1998": ("1998-07-17", "1998-10-08"),
    "Dot-com 2000-02": ("2000-03-24", "2002-10-09"),
    "GFC 2007-09": ("2007-10-09", "2009-03-09"),
    "Euro debt 2011": ("2011-04-29", "2011-10-03"),
    "China/oil 2015-16": ("2015-05-21", "2016-02-11"),
    "Q4 2018": ("2018-09-20", "2018-12-24"),
    "COVID 2020": ("2020-02-19", "2020-03-23"),
    "Rates 2022": ("2022-01-03", "2022-10-12"),
    "Tariffs 2025": ("2025-02-19", "2025-04-08"),
}

from dataclasses import replace
CFG_BASE = Config(k=SPEC["k"], lookback=SPEC["lookback_days"], rebalance_every=SPEC["rebalance_days"],
                  stress_q=SPEC["stress_q"], sub_len=SPEC["sub_len"], sub_step=SPEC["sub_step"],
                  agg_q=SPEC["agg_q"], kernel_a=SPEC["kernel_a"], cost_bps=SPEC["cost_bps"],
                  theta=0.5, n_random=N_RANDOM)
CFG_holdout = replace(CFG_BASE, test_start=SPEC["holdout_period"][0], test_end=SPEC["holdout_period"][1])
CFG_replication = replace(CFG_BASE, test_start=SPEC["replication_period"][0], test_end=None)
print("Holdout:", CFG_holdout.test_start, "to", CFG_holdout.test_end)
print("Replication:", CFG_replication.test_start, "to latest")

## 4. Holdout results, 1970–1989 (headline)

**How to read the main table.** `Proposed` = B-SR-DAS (primary); `Proposed-2` = S-SR-DAS (secondary); `Ablation` = the primary method with one ingredient changed; `Original` = the first specification (θ = 0.5) and pure diversity (θ = 0); `Baseline`/`Random`/`Benchmark` as before. Key columns: `StressCorr_OOS` (correlation of the held basket on market stress days, out of sample — lower is better), `Vol`, `MaxDD`, `StressBeta`, `Sharpe`.

### 4.1 Run the holdout backtest

In [ ]:
THETA_LOG.clear()
res_holdout = run_backtest(ind, MKT, v3_strategies(CFG_holdout), CFG_holdout)
theta_holdout = pd.Series(THETA_LOG).sort_index()
tab_holdout = summary_table(res_holdout, RF, market=MKT)
save_table(tab_holdout, "holdout_main_results")
display(format_table(tab_holdout))

#### Pre-registered hypotheses — holdout 1970–1989
`SUPPORTED` / `NOT supported` follows mechanically from the rules frozen in the specification cell.

In [ ]:
hyp_holdout = evaluate_hypotheses(res_holdout, "B-SR-DAS (balanced)", RF, B=N_BOOT)
save_table(hyp_holdout, "holdout_hypotheses_primary")
print("PRIMARY method: B-SR-DAS (balanced)")
display(hyp_holdout)
hyp2_holdout = evaluate_hypotheses(res_holdout, "S-SR-DAS (screen+diversify)", RF, B=N_BOOT, core_only=True)
save_table(hyp2_holdout, "holdout_hypotheses_secondary")
print("SECONDARY method: S-SR-DAS (screen+diversify)")
display(hyp2_holdout)

#### Sharpe-ratio comparisons with Holm correction — holdout 1970–1989
Positive `SharpeDiff_ann` = the primary method has the higher Sharpe ratio. Use `p_holm (bootstrap)` for claims of significance.

In [ ]:
sh_holdout = sharpe_tests_holm(res_holdout, "B-SR-DAS (balanced)", RF, B=N_BOOT)
save_table(sh_holdout, "holdout_sharpe_tests_holm")
display(sh_holdout.round(4))

#### Crisis behaviour and charts — holdout 1970–1989

In [ ]:
dd_holdout, ret_holdout = crisis_table(res_holdout, CRISES_HOLDOUT, MKT)
save_table(dd_holdout, "holdout_crisis_maxdd"); save_table(ret_holdout, "holdout_crisis_return")
print("Max drawdown inside each crisis window:")
display(dd_holdout.style.format("{:.1%}").background_gradient(cmap="RdYlGn", axis=0))
names = ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Original SR-DAS (theta=0.5)",
         "Original SR-DAS (theta=0)", "TailDep-Cluster", "Top-k Quality", "EW-All (1/N)"]
plot_wealth(res_holdout, names, f"Cumulative wealth — holdout 1970–1989")
plot_drawdowns(res_holdout, ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Top-k Quality", "EW-All (1/N)"],
               f"Drawdowns — holdout 1970–1989")
plot_metric_bars(tab_holdout, "StressCorr_OOS", f"Stress-day correlation of held basket, out of sample (lower = better) — holdout 1970–1989",
                 higher_is_better=False)

#### What θ did the balance rule choose? — holdout 1970–1989
The original specification fixed θ = 0.5. The rule picks θ from the data structure at each date (never from returns).

In [ ]:
print("balanced theta: mean %.3f, median %.3f, min %.3f, max %.3f" % (theta_holdout.mean(), theta_holdout.median(), theta_holdout.min(), theta_holdout.max()))
save_table(theta_holdout.to_frame("theta"), "holdout_balanced_theta")
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(theta_holdout.index, theta_holdout.values, color=PALETTE["Proposed"])
ax.axhline(0.5, ls="--", color=PALETTE["Original"], label="original fixed θ = 0.5")
ax.set_title(f"Balanced θ at each rebalance — holdout 1970–1989", loc="left"); ax.legend(frameon=False)
_style(ax); plt.tight_layout(); plt.show()

## 5. Replication, 1990–latest (supporting evidence)

This period was used while developing the revision, so it cannot confirm the method on its own. It shows whether the holdout conclusions also hold in the modern sample.

### 5.1 Run the replication backtest

In [ ]:
if RUN_REPLICATION:
    THETA_LOG.clear()
    res_replication = run_backtest(ind, MKT, v3_strategies(CFG_replication), CFG_replication)
    theta_replication = pd.Series(THETA_LOG).sort_index()
    tab_replication = summary_table(res_replication, RF, market=MKT)
    save_table(tab_replication, "replication_main_results")
    display(format_table(tab_replication))

#### Pre-registered hypotheses — replication 1990–latest
`SUPPORTED` / `NOT supported` follows mechanically from the rules frozen in the specification cell.

In [ ]:
if RUN_REPLICATION:
    hyp_replication = evaluate_hypotheses(res_replication, "B-SR-DAS (balanced)", RF, B=N_BOOT)
    save_table(hyp_replication, "replication_hypotheses_primary")
    print("PRIMARY method: B-SR-DAS (balanced)")
    display(hyp_replication)
    hyp2_replication = evaluate_hypotheses(res_replication, "S-SR-DAS (screen+diversify)", RF, B=N_BOOT, core_only=True)
    save_table(hyp2_replication, "replication_hypotheses_secondary")
    print("SECONDARY method: S-SR-DAS (screen+diversify)")
    display(hyp2_replication)

#### Sharpe-ratio comparisons with Holm correction — replication 1990–latest
Positive `SharpeDiff_ann` = the primary method has the higher Sharpe ratio. Use `p_holm (bootstrap)` for claims of significance.

In [ ]:
if RUN_REPLICATION:
    sh_replication = sharpe_tests_holm(res_replication, "B-SR-DAS (balanced)", RF, B=N_BOOT)
    save_table(sh_replication, "replication_sharpe_tests_holm")
    display(sh_replication.round(4))

#### Crisis behaviour and charts — replication 1990–latest

In [ ]:
if RUN_REPLICATION:
    dd_replication, ret_replication = crisis_table(res_replication, CRISES_REPLICATION, MKT)
    save_table(dd_replication, "replication_crisis_maxdd"); save_table(ret_replication, "replication_crisis_return")
    print("Max drawdown inside each crisis window:")
    display(dd_replication.style.format("{:.1%}").background_gradient(cmap="RdYlGn", axis=0))
    names = ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Original SR-DAS (theta=0.5)",
             "Original SR-DAS (theta=0)", "TailDep-Cluster", "Top-k Quality", "EW-All (1/N)"]
    plot_wealth(res_replication, names, f"Cumulative wealth — replication 1990–latest")
    plot_drawdowns(res_replication, ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Top-k Quality", "EW-All (1/N)"],
                   f"Drawdowns — replication 1990–latest")
    plot_metric_bars(tab_replication, "StressCorr_OOS", f"Stress-day correlation of held basket, out of sample (lower = better) — replication 1990–latest",
                     higher_is_better=False)

#### What θ did the balance rule choose? — replication 1990–latest
The original specification fixed θ = 0.5. The rule picks θ from the data structure at each date (never from returns).

In [ ]:
if RUN_REPLICATION:
    print("balanced theta: mean %.3f, median %.3f, min %.3f, max %.3f" % (theta_replication.mean(), theta_replication.median(), theta_replication.min(), theta_replication.max()))
    save_table(theta_replication.to_frame("theta"), "replication_balanced_theta")
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(theta_replication.index, theta_replication.values, color=PALETTE["Proposed"])
    ax.axhline(0.5, ls="--", color=PALETTE["Original"], label="original fixed θ = 0.5")
    ax.set_title(f"Balanced θ at each rebalance — replication 1990–latest", loc="left"); ax.legend(frameon=False)
    _style(ax); plt.tight_layout(); plt.show()

## 6. The quality–diversity trade-off frontier

SR-DAS with fixed θ from 0 (pure diversity) to 1 (mostly quality) traces a curve of *stress-day correlation* against *volatility* and against *max drawdown*. Other strategies appear as points. A good selection method sits **down and to the left** (diversified in stress *and* low risk). Points above the curve are dominated by some θ.

In [ ]:
THETAS = [0.0, 0.025, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]
if RUN_FRONTIER:
    samples = [("holdout", CFG_holdout, "tab_holdout")]
    if RUN_REPLICATION:
        samples.append(("replication", CFG_replication, "tab_replication"))
    for tag, cfg, tabname in samples:
        print(f"frontier: {tag} ...")
        front = run_frontier(ind, MKT, cfg, THETAS, RF)
        save_table(front, f"{tag}_frontier")
        tab = globals()[tabname]
        pts = tab[~tab["Group"].isin(["Ablation"])]
        display(front[["theta", "CAGR", "Vol", "Sharpe", "MaxDD", "StressCorr_OOS", "StressBeta"]].round(4))
        plot_frontier(front, pts, "StressCorr_OOS", "Vol", f"Stress-day correlation vs volatility — {tag}")
        plot_frontier(front, pts, "StressCorr_OOS", "MaxDD", f"Stress-day correlation vs max drawdown — {tag}")

## 7. Summary of verdicts
One table with every pre-registered hypothesis in both samples. The holdout column is the evidence for the paper's claims; the replication column shows consistency.

In [ ]:
fmt = lambda h: h["Verdict"] + "  (" + h["Statistic"] + "; " + h["Detail"] + ")"
summary = pd.DataFrame({"Description": hyp_holdout["Description"], "Holdout 1970-1989": fmt(hyp_holdout)})
if RUN_REPLICATION:
    summary["Replication 1990+"] = fmt(hyp_replication)
save_table(summary, "verdict_summary_primary")
display(summary)
sec = pd.DataFrame({"Description": hyp2_holdout["Description"], "Holdout 1970-1989": hyp2_holdout["Verdict"]})
if RUN_REPLICATION:
    sec["Replication 1990+"] = hyp2_replication["Verdict"]
save_table(sec, "verdict_summary_secondary")
print("Secondary method (S-SR-DAS):")
display(sec)
print("Specification fingerprint:", SPEC_HASH)

In [ ]:
# Daily returns of all strategies (for any further analysis), then zip everything for download
pd.DataFrame({n: r.returns for n, r in res_holdout.items()}).to_csv(os.path.join(OUT_DIR, "holdout_daily_returns.csv"))
if RUN_REPLICATION:
    pd.DataFrame({n: r.returns for n, r in res_replication.items()}).to_csv(os.path.join(OUT_DIR, "replication_daily_returns.csv"))
import shutil
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("Saved:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download(OUT_DIR + ".zip")
except Exception:
    print("Not in Colab: the archive is", OUT_DIR + ".zip")

## 8. Notes for the write-up

* Report the holdout verdicts as they come out, including `NOT supported` ones. The specification fingerprint shows that the hypotheses were fixed before the holdout was run.
* The 1990+ replication was seen during development: describe it as consistency evidence, not confirmation.
* Sharpe differences between selection methods are usually not significant after Holm correction; the method's claims should rest on the diversification and risk hypotheses (H1–H3, H5–H7).